Here I try this idea on fitting drift-diffussion models (Ratclif) to decision processes in chess games. This is work in progress

In [5]:
# ── 0. Install dependencies ──────────────────────────────────────────
!pip install python-chess zstandard -q

# ── 1. Mount Drive for persistent storage ───────────────────────────
from google.colab import drive
SAVE_PATH = '/content/drive/MyDrive/ChessDDM/pilot.parquet'


In [6]:
import os

path_to_check = '/content/drive/MyDrive/ChessDDM/pilot.parquet' # Example path

if os.path.exists(path_to_check):
    print(f"The path '{path_to_check}' exists.")
elif os.path.isdir(os.path.dirname(path_to_check)):
    print(f"The directory for '{path_to_check}' exists, but the file does not.")
else:
    print(f"The path '{path_to_check}' does not exist, and neither does its directory.")

The path '/content/drive/MyDrive/ChessDDM/pilot.parquet' does not exist, and neither does its directory.


In [7]:
import os

directory_to_create = '/content/drive/MyDrive/ChessDDM/'

os.makedirs(directory_to_create, exist_ok=True)
print(f"Directory '{directory_to_create}' created successfully or already exists.")

Directory '/content/drive/MyDrive/ChessDDM/' created successfully or already exists.


In [ ]:

# ── 2. Stream + parse ────────────────────────────────────────────────
import zstandard as zstd
import chess.pgn
import io, requests, pandas as pd
from tqdm import tqdm

# Set pandas display option for float format to show more digits
pd.set_option('display.float_format', '{:.3f}'.format)

URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2024-10.pgn.zst"

TARGET_GAMES  = 5_000   # games WITH eval — stop here
MAX_GAMES_SCANNED = 10_000  # safety cap

records = []
games_with_eval = 0
games_scanned   = 0

with requests.get(URL, stream=True) as r:
    dctx = zstd.ZstdDecompressor()

    with dctx.stream_reader(r.raw) as reader:
        text_stream = io.TextIOWrapper(reader, encoding='utf-8')

        while games_with_eval < TARGET_GAMES and games_scanned < MAX_GAMES_SCANNED:
            game = chess.pgn.read_game(text_stream)
            if game is None:
                break

            games_scanned += 1
            headers = game.headers

            # ── Basic filters ──
            time_ctrl = headers.get("TimeControl", "")
            white_elo = headers.get("WhiteElo",   "?")
            black_elo = headers.get("BlackElo",   "?")
            termination = headers.get("Termination", "")
            result      = headers.get("Result", "*")   # "1-0", "0-1", "1/2-1/2"

            if white_elo == "?" or black_elo == "?":
                continue
            if "Bot" in headers.get("WhiteTitle","") or \
               "Bot" in headers.get("BlackTitle",""):
                continue

            # ── Extract move-level data ──
            node = game
            move_records = []
            move_num = 0
            prev_eval = None
            prev_clk  = {True: None, False: None}  # True=white, False=black

            while not node.is_end():
                next_node = node.variation(0)
                move_num += 1

                is_white = (move_num % 2 == 1)

                # Clock — essential for move-time; skip the move if missing
                clk = next_node.clock()           # seconds, or None
                if clk is None:
                    node = next_node
                    prev_clk[is_white] = clk
                    continue

                # Move time = previous clock - current clock
                move_time = None
                if prev_clk[is_white] is not None:
                    move_time = prev_clk[is_white] - clk

                # Eval (always from White's POV).
                # Terminal positions (checkmate) carry no %eval tag — store as
                # None and let R impute via fill(). Mate-in-N scores are capped
                # at ±1000 so they stay on the same scale as regular evals.
                ev = next_node.eval()             # chess.engine.PovScore or None
                if ev is None:
                    cp = None                     # last move of game (terminal)
                elif ev.white().is_mate():
                    cp = 1000 if ev.white().mate() > 0 else -1000
                else:
                    cp = max(-1000, min(1000, ev.white().score()))

                move_records.append({
                    'move_num':   move_num,
                    'is_white':   is_white,
                    'move_time':  move_time,    # RT proxy
                    'eval_cp':    cp,            # position eval after move
                    'clock_left': clk,           # remaining clock
                })

                prev_clk[is_white] = clk
                prev_eval = cp
                node = next_node

            if not move_records or all(r['eval_cp'] is None for r in move_records):
                continue  # no eval in this game, skip

            # ── Game-level features ──
            white_wins = 1 if result == "1-0" else (0 if result == "0-1" else None)
            for r in move_records:
                r['white_elo']    = int(white_elo)
                r['black_elo']    = int(black_elo)
                r['time_control'] = time_ctrl
                r['game_id']      = headers.get("Site","").split("/")[-1]
                r['termination']  = termination
                r['white_wins']   = white_wins

            records.extend(move_records)
            games_with_eval += 1

            if games_with_eval % 500 == 0:
                print(f"  {games_with_eval} eval games | {games_scanned} scanned")

# ── 3. Save ──────────────────────────────────────────────────────────
df = pd.DataFrame(records)
df.to_parquet(SAVE_PATH, index=False)
print(f"\nSaved {len(df):,} move rows from {games_with_eval} games → {SAVE_PATH}")
print(df.head())
